Interpretation of *in vitro* data and **DiaMOND** profiles from a holistic point of view is complicated looking at sequential modeling. While sequential modeling has an intuitive logic flow, trying to interpret individual features is not as straightforward, due to the time-dependence of the model, and the lack of linear-time-responsiveness from drug combinations. To better illustrate the _full impact_ of treatment, without as much emphasis on time dependency, a more simple model is to be created, projecting only the terminal timepoint (week 8, timepoint 6) from the starting lesion information and the in vitro data.

In [1]:
# imports

import polars as pl
import numpy as np
import datetime
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
import joblib


from data.MarmosetData import MarmosetData
from data.InVitroData import DiamondData

from helpers.PlottingFunctions import (
    plot_lesion_progressions,
    plot_performance_metrics,
    plot_feature_importance_bars,
    plot_shap_summary_dot
)
from helpers.ModelingFunctions import (
    impute_within_compound,
    split_by_compound,
)
from helpers.FeatureAnalysisFunctions import (
    get_rf_feature_importance,
    calculate_shap_values,
    get_permutation_importance,
)

In [3]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 10

SAVE_MODELS = False
SAVE_MODEL_RESULTS = False
SAVE_FIGURES = False

MODEL_DIR = "modeling/minimal/models/"
MODEL_RESULTS_DIR = "modeling/minimal/results/"
FIGURE_SAVE_DIR = "figures/minimal/"
FIGURE_SAVE_FORMAT = "svg"

np.random.seed(RANDOM_STATE)
now = datetime.datetime.now().strftime("%Y%m%d")

In [ ]:
# data processing
marmoset_data = MarmosetData("data/marm_data_wide_clustered_classif.csv")
diamond_data = DiamondData("data/in_vitro_diamond_data.csv")

# extract severe lesions
severe_lesions = marmoset_data.get_severe_lesions().data

# targets
x_features_marm = ["TP2_MeanHU", "TP2_MeanSUV"]
y_features = ["TP6_MeanHU", "TP6_MeanSUV"]
metadata_features = ["Compound", "Lesion"]

severe_data = severe_lesions.select(x_features_marm + y_features + metadata_features)

diamond_df = diamond_data.data.filter(
    pl.col("NumbDrugs") > 1
    ).drop("NumbDrugs")
combo_features = [col for col in diamond_df.columns if "Drug" not in col]

severe_merged = severe_data.join(
    other=diamond_df,
    left_on="Compound",
    right_on="Drug",
    how="inner"
)

severe_merged.head()



In [17]:
severe_imputed = impute_within_compound(severe_merged, columns=severe_merged.columns, grouping_column="Compound")
train, test = split_by_compound(severe_imputed, grouping_column="Compound", test_size=0.2, random_state=42)


In [18]:
base_input_features = [f for f in severe_merged.columns if not f.startswith("TP6") and f not in metadata_features]
base_output_features = y_features

X_train = train.select(base_input_features)
X_test = test.select(base_input_features)

y_train = train.select(base_output_features)
y_test = test.select(base_output_features)

In [41]:
model = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=42
)

model.fit(X_train, y_train)


RandomForestRegressor(max_depth=10, random_state=42)

In [ ]:
predictions = model.predict(X_test)

test_pred_df = pl.DataFrame()
test_pred_df = test_pred_df.with_columns(
    test["Compound"],
    test["Lesion"]
)

for i, col_name in enumerate(y_features):
    test_pred_df = test_pred_df.with_columns(
        pl.Series(col_name, predictions[:, i])
    )

In [47]:
metrics = []
for output_name in y_features:
    y_true = test.select(output_name).to_numpy().flatten()
    y_pred = test_pred_df.select(output_name).to_numpy().flatten()
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    metrics.append({
        "output": output_name,
        "mse": mse,
        "mae": mae,
        "r2": r2
    })
    
metrics_df = pl.DataFrame(metrics)
print(metrics_df)

shape: (2, 4)
┌─────────────┬─────────────┬───────────┬───────────┐
│ output      ┆ mse         ┆ mae       ┆ r2        │
│ ---         ┆ ---         ┆ ---       ┆ ---       │
│ str         ┆ f64         ┆ f64       ┆ f64       │
╞═════════════╪═════════════╪═══════════╪═══════════╡
│ TP6_MeanHU  ┆ 9670.051975 ┆ 79.107463 ┆ -0.044444 │
│ TP6_MeanSUV ┆ 0.628616    ┆ 0.573211  ┆ 0.117374  │
└─────────────┴─────────────┴───────────┴───────────┘
